In [ ]:
%pip install segmentation-models-pytorch albumentations opencv-python pandas tqdm scikit-learn seaborn -q

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
%pip install --upgrade ipywidgets widgetsnbextension jupyterlab_widgets  -q

In [ ]:
import os, cv2, torch, numpy as np, pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torch import nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")  

In [ ]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.endswith(".png")])
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = None

        if self.mask_dir:
            mask = cv2.imread(os.path.join(self.mask_dir, fname), cv2.IMREAD_GRAYSCALE)

        if self.transform:
            if mask is not None:
                augmented = self.transform(image=img, mask=mask)
                img, mask = augmented["image"], augmented["mask"]
            else:
                augmented = self.transform(image=img)
                img = augmented["image"]

        return (img, mask.long()) if mask is not None else (img, fname)

In [ ]:
train_tf = A.Compose(
    [
        # First resize to standard size to avoid crop errors
        A.Resize(576, 576),
        # Spatial augmentations
        A.RandomCrop(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=20, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, p=0.3),
        # Color augmentations
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.GaussNoise(p=0.2),
        A.GaussianBlur(blur_limit=3, p=0.2),
        # Weather augmentations for robustness
        A.RandomRain(p=0.1),
        A.RandomFog(p=0.1),
        # Normalization
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    keypoint_params=None,
)

val_tf = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

test_tf = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

In [ ]:
train_img_dir = "./data/train/imgs"
train_mask_dir = "./data/train/masks"
test_img_dir = "./data/test/imgs"

# Get all training filenames
all_imgs = sorted([f for f in os.listdir(train_img_dir) if f.endswith(".png")])

# Split into train and validation (80-20)
train_imgs, val_imgs = train_test_split(all_imgs, test_size=0.2, random_state=42)

print(f"Train images: {len(train_imgs)}, Validation images: {len(val_imgs)}")

# Create dataloaders
train_set = SegDataset(train_img_dir, train_mask_dir, transform=train_tf)
train_loader = DataLoader(
    train_set, batch_size=8, shuffle=True, num_workers=2, pin_memory=True
)

val_set = SegDataset(train_img_dir, train_mask_dir, transform=val_tf)
val_loader = DataLoader(
    val_set, batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)

test_set = SegDataset(test_img_dir, transform=test_tf)
test_loader = DataLoader(test_set, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# model = smp.DeepLabV3Plus(
#     encoder_name="efficientnet-b5",
#     encoder_weights="imagenet",
#     classes=16,
#     activation=None,
# ).to(DEVICE)

# Option 2: FPN (Faster, good accuracy)
# model = smp.FPN(
#     encoder_name="efficientnet-b4",
#     encoder_weights="imagenet",
#     classes=16,
#     activation=None
# ).to(DEVICE)

# Option 3: U-Net++ (Improved U-Net)
model = smp.UnetPlusPlus(
    encoder_name="resnet101",
    encoder_weights="imagenet",
    classes=16,
    activation=None
).to(DEVICE)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        target_one_hot = torch.zeros_like(pred)
        target_one_hot.scatter_(1, target.unsqueeze(1), 1)

        intersection = (pred * target_one_hot).sum()
        union = pred.sum() + target_one_hot.sum()
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        ce_loss = nn.functional.cross_entropy(pred, target, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# Combine multiple losses for better convergence
criterion_ce = nn.CrossEntropyLoss()
criterion_dice = DiceLoss()
criterion_focal = FocalLoss()


def combined_loss(pred, target):
    ce = criterion_ce(pred, target)
    dice = criterion_dice(pred, target)
    focal = criterion_focal(pred, target)
    return 0.5 * ce + 0.3 * dice + 0.2 * focal

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

In [ ]:
def calculate_iou(pred, target, num_classes=16):
    """Calculate IoU for each class"""
    ious = []
    pred = pred.argmax(dim=1)

    for c in range(num_classes):
        pred_c = (pred == c).float()
        target_c = (target == c).float()

        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum() - intersection

        iou = (intersection + 1e-6) / (union + 1e-6)
        ious.append(iou.item())

    return np.mean(ious)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_iou = 0

    for imgs, masks in tqdm(loader, desc="Training"):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, masks)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_iou += calculate_iou(out, masks)

    return total_loss / len(loader), total_iou / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_iou = 0

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Validating"):
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            loss = criterion(out, masks)

            total_loss += loss.item()
            total_iou += calculate_iou(out, masks)

    return total_loss / len(loader), total_iou / len(loader)

In [ ]:
EPOCHS = 50
best_iou = 0
patience = 10
patience_counter = 0

train_losses, val_losses = [], []
train_ious, val_ious = [], []

for epoch in range(EPOCHS):
    train_loss, train_iou = train_epoch(
        model, train_loader, optimizer, combined_loss, DEVICE
    )
    val_loss, val_iou = validate(model, val_loader, combined_loss, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ious.append(train_iou)
    val_ious.append(val_iou)

    print(
        f"[Epoch {epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f}, Train IoU: {train_iou:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}"
    )

    scheduler.step()

    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_model.pth")
        patience_counter = 0
        print(f"✅ Best model saved with IoU: {val_iou:.4f}")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load("best_model.pth"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label="Train Loss", marker="o")
axes[0].plot(val_losses, label="Val Loss", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_ious, label="Train IoU", marker="o")
axes[1].plot(val_ious, label="Val IoU", marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].set_title("Training and Validation mIoU")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best validation IoU: {max(val_ious):.4f}")

In [ ]:
def rle_encode(mask):
    pixels = mask.flatten(order="F").astype(np.uint8)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[:-1:2]
    return " ".join(map(str, runs))

In [ ]:
model.eval()
records = []
orig_h, orig_w = 1024, 1024  # UAV test image original size

with torch.no_grad():
    for imgs, fnames in tqdm(test_loader, desc="Generating predictions"):
        imgs = imgs.to(DEVICE)
        preds = model(imgs).softmax(1).argmax(1).cpu().numpy()

        for pred, fname in zip(preds, fnames):
            pred = cv2.resize(
                pred.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST
            )
            row = {"img": fname}
            for c in range(16):
                m = (pred == c).astype(np.uint8)
                row[f"class_{c}"] = "none" if m.sum() == 0 else rle_encode(m)
            records.append(row)

sub = pd.DataFrame(records)
sub.to_csv("submission.csv", index=False)
print("✅ submission.csv saved!")

In [ ]:
import numpy as np, cv2, os
from tqdm import tqdm

train_mask_dir = "./data/train/masks"
unique_labels = set()

for f in tqdm(os.listdir(train_mask_dir)):
    mask = cv2.imread(os.path.join(train_mask_dir, f), cv2.IMREAD_GRAYSCALE)
    unique_labels.update(np.unique(mask).tolist())

print("All unique labels in training set:", sorted(unique_labels))

In [ ]:
df = pd.read_csv("submission.csv")
print(df.head())
print(df.columns)

In [ ]:
mask = cv2.imread(os.path.join(train_mask_dir, "3999.png"), cv2.IMREAD_GRAYSCALE)
print(np.unique(mask))

In [ ]:
!kaggle competitions submit -c 2025-ncku-ee-ml-16-classes-segmentation -f submission.csv -m "resize Enhanced DeepLabV3+ with multi-loss"